# Bandwidth
**Bandwidth** is a critical factor in Federated Learning systems as it determines the communication overhead between the central server and distributed clients. In each communication round, model parameters must be transmitted over the network - clients send their updated model weights to the server (upload), and the server broadcasts the aggregated global model back to clients (download).

### Why Bandwidth Matters

1. **Communication Efficiency**: Large models (like LLMs) can be hundreds of megabytes or gigabytes. Repeated transmission of these parameters across multiple rounds can lead to significant bandwidth consumption.

2. **Latency**: Bandwidth limitations directly impact training time. Higher bandwidth enables faster communication rounds, reducing overall training duration.

3. **Scalability**: As the number of clients increases, bandwidth requirements grow linearly. This becomes a bottleneck for scaling federated systems.

4. **Client Heterogeneity**: Clients may have varying network capabilities (mobile devices vs. high-speed data centers). Bandwidth optimization techniques help accommodate diverse client conditions.

### Bandwidth Optimization Techniques

- **Model Compression**: Techniques like quantization and pruning reduce model size before transmission
- **Gradient Sparsification**: Only sending important gradient updates instead of full weights
- **Sketching & Low-rank Updates**: Compressing updates using dimensionality reduction
- **Adaptive Communication**: Adjusting communication frequency based on gradient staleness

In this notebook, we demonstrate how to track and measure the bandwidth consumed during federated learning rounds using Flower.


In [12]:
import logging
import warnings
from logging import INFO, ERROR, LogRecord
from collections import OrderedDict
from typing import Dict, List, Optional, Tuple, Union

from flwr.common import (
    Context,
    EvaluateRes,
    FitIns,
    FitRes,
    MessageType,
    Parameters,
    Scalar,
    Context,
    parameters_to_ndarrays,
    ndarrays_to_parameters,
)
from flwr.common.logger import (
    ConsoleHandler,
    console_handler,
    FLOWER_LOGGER,
    LOG_COLORS,
    log, update_console_handler,
)
from flwr.client import Client, ClientApp, NumPyClient
from flwr.client.mod import arrays_size_mod

from flwr.server import ServerAppComponents, ClientManager, ServerApp, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation


import torch
from transformers import AutoModelForCausalLM, GPTNeoXForCausalLM


In [16]:
# Customize logging for the course.
class InfoFilter(logging.Filter):
    def filter(self, record):
        return record.levelno == INFO

warnings.filterwarnings("ignore")

FLOWER_LOGGER.removeHandler(console_handler)

# To filter logging coming from the Simulation Engine
# so it's more readable in notebooks
backend_setup = {"init_args": {"logging_level": ERROR, "log_to_driver": True}}

class ConsoleHandlerV2(ConsoleHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def format(self, record: LogRecord) -> str:
        """Format function that adds colors to log level."""
        if self.json:
            log_fmt = "{lvl='%(levelname)s', time='%(asctime)s', msg='%(message)s'}"
        else:
            log_fmt = (
                f"{LOG_COLORS[record.levelname] if self.colored else ''}"
                f"%(levelname)s {'%(asctime)s' if self.timestamps else ''}"
                f"{LOG_COLORS['RESET'] if self.colored else ''}"
                f": %(message)s"
            )
        formatter = logging.Formatter(log_fmt)
        return formatter.format(record)


console_handlerv2 = ConsoleHandlerV2(
    timestamps=False,
    json=False,
    colored=True,
)
console_handlerv2.setLevel(INFO)
console_handlerv2.addFilter(InfoFilter())
FLOWER_LOGGER.addHandler(console_handlerv2)

### Define the model
We initialize the **Pythia-14m** model from EleutherAI. This is a small language model designed for research purposes with approximately 14 million parameters. The model size directly impacts bandwidth consumption during federated learning - larger models require more bandwidth to transmit between server and clients.

*  Find more information about [EleutherAI/pythia-14m](https://huggingface.co/EleutherAI/pythia-14m)

In [3]:
model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/pythia-14m",
    cache_dir="./pythia-14m/cache",
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [4]:
# Get some Model values.
vals = model.state_dict().values()
total_size_bytes = sum(p.element_size() * p.numel() for p in vals)
total_size_mb = int(total_size_bytes / (1024**2))

log(INFO, "Model size is: {} MB".format(total_size_mb))

INFO : Model size is: 26 MB


In [5]:
def set_weights(net, parameters):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
    net.load_state_dict(state_dict, strict=False)

def get_weights(net):
    ndarrays = [
        val.cpu().numpy() for _, val in net.state_dict().items()
    ]
    return ndarrays

### Define the FlowerClient.
The **FlowerClient** class handles local training and evaluation on each client device. In this example, we implement a minimal client that:
- **fit()**: Receives global model parameters, sets local weights, and returns updated weights (no actual training performed)
- **evaluate()**: Receives global model parameters, sets local weights, and returns evaluation metrics

In real federated learning scenarios, this is where local training on client data would occur.


In [6]:
class FlowerClient(NumPyClient):
    def __init__(self, net):
        self.net = net

    def fit(self, parameters, config):
        set_weights(self.net, parameters)
        # No actual training here
        return get_weights(self.net), 1, {}

    def evaluate(self, parameters, config):
        set_weights(self.net, parameters)
        # No actual evaluation here
        return float(0), int(1), {"accuracy": 0}


def client_fn(context: Context) -> FlowerClient:
    return FlowerClient(model).to_client()


client = ClientApp(
    client_fn,
    mods=[arrays_size_mod],
)

### Define the custom strategy: BandwidthTrackingFedAvg.
We extend the **FedAvg** strategy with bandwidth tracking capabilities. The key methods we override are:

- **aggregate_fit()**: Called when the server receives model updates from clients. Here we track the size of each model received from clients (upload bandwidth from client perspective).

- **configure_fit()**: Called when the server sends instructions to clients. Here we track the size of the global model being sent to clients (download bandwidth from client perspective).

By instrumenting these methods, we can measure the total bandwidth consumed during each federated learning round.


In [7]:
bandwidth_sizes = []


class BandwidthTrackingFedAvg(FedAvg):
    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        # Track sizes of models received
        for _, res in results:
            ndas = parameters_to_ndarrays(res.parameters)
            size = int(sum(n.nbytes for n in ndas) / (1024**2))
            log(INFO, f"Server receiving model size: {size} MB")
            bandwidth_sizes.append(size)

        # Call FedAvg for actual aggregation
        return super().aggregate_fit(server_round, results, failures)

    def configure_fit(self, server_round, parameters, client_manager):
        # Call FedAvg for actual configuration
        instructions = super().configure_fit(
            server_round, parameters, client_manager
        )

        # Track sizes of models to be sent
        for _, ins in instructions:
            ndas = parameters_to_ndarrays(ins.parameters)
            size = int(sum(n.nbytes for n in ndas) / (1024**2))
            log(INFO, f"Server sending model size: {size} MB")
            bandwidth_sizes.append(size)

        return instructions

In [8]:
params = ndarrays_to_parameters(get_weights(model))

def server_fn(context: Context):
    strategy = BandwidthTrackingFedAvg(
        fraction_evaluate=0.0,
        initial_parameters=params,
    )
    config = ServerConfig(num_rounds=1)
    return ServerAppComponents(
        strategy=strategy,
        config=config,
    )


server = ServerApp(server_fn=server_fn)

### Run the simulation
We execute the federated learning simulation with:
- **2 supernodes (clients)**: Simulating distributed devices participating in federated learning
- **1 communication round**: A single round of parameter exchange
- **Backend configuration**: Logging level set to ERROR to reduce noise and show only important information

The simulation demonstrates the bandwidth tracking in action.


In [9]:
run_simulation(server_app=server,
               client_app=client,
               num_supernodes=2,
               backend_config=backend_setup
)

INFO : Starting Flower ServerApp, config: num_rounds=1, no round_timeout
INFO : 
INFO : [INIT]
INFO : Using initial global parameters provided by strategy
INFO : Starting evaluation of initial global parameters
INFO : Evaluation returned no results (`None`)
INFO : 
INFO : [ROUND 1]
INFO : Server sending model size: 26 MB
INFO : Server sending model size: 26 MB
INFO : configure_fit: strategy sampled 2 clients (out of 2)
/home/sanjeev/Templates/old-projects/Machine-learning-Project/.venv/lib/python3.10/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(ClientAppActor pid=84140) INFO :      Incoming `ArrayRecord` size statistics:
(ClientAppActor pid=84140) INFO :      {'fitins.parameters': {'elements': 14067712, 'bytes': 28145294}}


### Log how much bandwidth was used!
After the simulation completes, we can see the total bandwidth consumed:
- **Model sizes sent to clients** (server → client): The global model is broadcast to all participating clients
- **Model sizes received from clients** (client → server): Each client sends back their locally trained model updates

The sum of all these transmissions represents the total bandwidth usage for the federated learning round. This metric is crucial for:
- Estimating infrastructure costs
- Planning for production deployments
- Optimizing communication protocols


In [10]:
log(INFO, "Total bandwidth used: {} MB".format(sum(bandwidth_sizes)))

INFO : Total bandwidth used: 104 MB
